In [ ]:
# --- Attach Density-Matrix Attention to selected transformer layers ---
import math
import torch
import torch.nn as nn

class _DMState(torch.nn.Module):
    """Holds per-layer hyperparams + exposes a buffer for training-time logging."""
    def __init__(self, alpha=0.5, lambda_coh=0.0, symmetrize=True, trace_norm=True):
        super().__init__()
        self.alpha = nn.Parameter(torch.tensor(float(alpha)))  # learnable [0..1] if you like; or freeze by .requires_grad_(False)
        self.lambda_coh = float(lambda_coh)
        self.symmetrize = bool(symmetrize)
        self.trace_norm = bool(trace_norm)
        self.register_buffer("last_coh_loss", torch.zeros(()), persistent=False)

def _density_from_scores(scores, mask=None, symmetrize=True, trace_norm=True, eps=1e-8):
    # scores: [B, H, Tq, Tk]
    A = torch.softmax(scores, dim=-1)                      # vanilla attn weights
    if symmetrize:
        # Make it closer to a correlation-like matrix head-wise
        A = 0.5 * (A + A.transpose(-1, -2))                # [B,H,T,T], assumes Tq==Tk (true for self-attn)
    if mask is not None:
        # optional: enforce padding mask
        A = A * mask                                       # broadcast-safe if mask is [B,1,T,T]
    if trace_norm:
        tr = A.diagonal(dim1=-2, dim2=-1).sum(-1, keepdim=True).unsqueeze(-1)  # [B,H,1,1]
        A = A / (tr + eps)
    return A

def _coherence_L1(rho):
    # L1 off-diagonals: ||rho - diag(rho)||_1
    diag = rho.diagonal(dim1=-2, dim2=-1)
    diag_mat = torch.zeros_like(rho)
    diag_mat = diag_mat + torch.diag_embed(diag)
    return (rho - diag_mat).abs().sum(dim=(-1, -2)).mean()

def _wrap_attn_with_dma(attn_mod, dmstate: _DMState):
    """
    Replace attn_mod.forward with a DMA version.
    Works for Qwen/LLaMA-like attn blocks that expose q_proj/k_proj/v_proj/o_proj and rotary.
    Falls back to vanilla forward if unexpected kwargs show up.
    """
    orig_forward = attn_mod.forward
    q_proj = getattr(attn_mod, "q_proj", None)
    k_proj = getattr(attn_mod, "k_proj", None)
    v_proj = getattr(attn_mod, "v_proj", None)
    o_proj = getattr(attn_mod, "o_proj", None)
    num_heads = getattr(attn_mod, "num_heads", None)
    head_dim = getattr(attn_mod, "head_dim", None)

    if not all([q_proj, k_proj, v_proj, o_proj, num_heads, head_dim]):
        # If structure doesn’t match, keep original
        return

    rotary_emb = getattr(attn_mod, "rotary_emb", None)  # present in Qwen/LLaMA
    scale = 1.0 / math.sqrt(head_dim)

    def split_heads(x):
        # x: [B, T, D] -> [B, H, T, d]
        B, T, D = x.shape
        x = x.view(B, T, num_heads, head_dim).transpose(1, 2)
        return x

    def merge_heads(x):
        # x: [B, H, T, d] -> [B, T, D]
        B, H, T, d = x.shape
        x = x.transpose(1, 2).contiguous().view(B, T, H * d)
        return x

    def dma_forward(hidden_states, attention_mask=None, position_ids=None, past_key_value=None,
                    output_attentions=False, use_cache=False, **kwargs):
        # hidden_states: [B, T, D]
        q = q_proj(hidden_states)
        k = k_proj(hidden_states)
        v = v_proj(hidden_states)

        q = split_heads(q)   # [B,H,T,d]
        k = split_heads(k)   # [B,H,T,d]
        v = split_heads(v)   # [B,H,T,d]

        # rotary (if present)
        if rotary_emb is not None and position_ids is not None:
            # Qwen/LLaMA rotary apply function name may vary:
            # some impls attach 'apply_rotary_pos_emb' or pack inside rotary_emb itself
            apply_rotary = getattr(attn_mod, "apply_rotary_pos_emb", None)
            if apply_rotary is not None:
                q, k = apply_rotary(q, k, position_ids)
            else:
                # common rotary_emb(q, k, position_ids) signature
                try:
                    q, k = rotary_emb(q, k, position_ids)
                except:
                    pass

        scores = torch.matmul(q, k.transpose(-1, -2)) * scale  # [B,H,T,T]

        # Build an attention mask in [B,1,T,T] form if provided (HF uses additive mask with -inf on blocked positions)
        attn_mask_mult = None
        if attention_mask is not None and attention_mask.dim() == 4:
            # already [B,1,Tq,Tk] in some HF impls as additive; convert to multiplicative 0/1
            attn_mask_mult = (attention_mask == 0).to(scores.dtype)  # or infer by threshold
            attn_mask_mult = 1.0 - attn_mask_mult
        # Vanilla weights for baseline path
        A_soft = torch.softmax(scores + (attention_mask if (attention_mask is not None and attention_mask.dim()==4) else 0), dim=-1)

        # Density-matrix path
        rho = _density_from_scores(scores, mask=attn_mask_mult, symmetrize=dmstate.symmetrize, trace_norm=dmstate.trace_norm)

        # Mix: out = alpha * (A_soft V) + (1-alpha) * (rho V)
        out_soft = torch.matmul(A_soft, v)
        out_rho  = torch.matmul(rho,    v)
        alpha = torch.clamp(dmstate.alpha, 0.0, 1.0)
        context = alpha * out_soft + (1.0 - alpha) * out_rho

        # Optional coherence penalty for training
        if dmstate.lambda_coh > 0.0 and context.requires_grad:
            dmstate.last_coh_loss = _coherence_L1(rho) * dmstate.lambda_coh
        else:
            dmstate.last_coh_loss = dmstate.last_coh_loss.detach() * 0.0

        # merge heads and project out
        context = merge_heads(context)
        output = o_proj(context)

        if output_attentions:
            # return the density weights so you can visualize or debug
            return output, rho
        return output, None if not use_cache else past_key_value

    # Disable kernel shortcuts for these layers to ensure Python path is used
    for flag_name in ["use_flash_attn", "use_sdpa", "use_fused_attn"]:
        if hasattr(attn_mod, flag_name):
            setattr(attn_mod, flag_name, False)

    attn_mod._dmstate = dmstate
    attn_mod.forward = dma_forward

def patch_qwen3vl_dma(model, layers="first_k", k=6, alpha=0.5, lambda_coh=0.0, symmetrize=True, trace_norm=True):
    """
    model: your Unsloth-wrapped Qwen-3-VL model (torch.nn.Module)
    layers: "first_k" | "last_k" | list of indices
    """
    # Try to unwrap Unsloth to HF backbone where .model.layers exist
    base = model
    for name in ["model", "language_model", "base_model", "transformer"]:
        if hasattr(base, name):
            base = getattr(base, name)

    # Find the stack of decoder layers
    stack = None
    for candidate in ["layers", "h"]:
        if hasattr(base, candidate):
            stack = getattr(base, candidate)
            break
    if stack is None:
        raise RuntimeError("Could not find transformer layer stack on this model.")

    n_layers = len(stack)
    if isinstance(layers, str):
        if layers == "first_k":
            target = list(range(min(k, n_layers)))
        elif layers == "last_k":
            target = list(range(max(0, n_layers - k), n_layers))
        else:
            raise ValueError("layers must be 'first_k', 'last_k', or a list of indices")
    else:
        target = list(layers)

    attached = []
    for idx in target:
        block = stack[idx]
        # Qwen/LLaMA style names: .self_attn or .attention
        attn = None
        for name in ["self_attn", "attention", "attn"]:
            if hasattr(block, name):
                attn = getattr(block, name)
                break
        if attn is None:
            continue
        dmstate = _DMState(alpha=alpha, lambda_coh=lambda_coh, symmetrize=symmetrize, trace_norm=trace_norm)
        _wrap_attn_with_dma(attn, dmstate)
        attached.append(idx)

    return attached  # list of layer indices actually patched

# --- Example usage ---
# model is your Unsloth-loaded Qwen-3-VL
# attached_layers = patch_qwen3vl_dma(model, layers="first_k", k=8, alpha=0.6, lambda_coh=1e-3)
# print("DMA attached to layers:", attached_layers)
